# PricePoint — 02 · Linear Regression

**What it is:** the simplest model — one straight line: it multiplies every feature by a weight, adds them up, and predicts a price.

**Why we start here:** fast, easy to understand, and it is our **baseline**. The other three models should beat it.

**Next:** train it, score it, save the scores.

In [1]:
import json

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [2]:
# --- file paths -------------------------------------------------------

if Path.cwd().name == "notebooks":
    DATA_DIR = Path("../data")
    MODEL_DIR = Path("../models")
else:
    DATA_DIR = Path("data")
    MODEL_DIR = Path("models")

In [3]:
# --- load the clean data ---------------------------------------------

clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")

# True / False -> 1 / 0 so the models can use them
clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

# the columns the models will use
categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("rows:", len(X))
print("features:", X.shape[1])

rows: 1303
features: 14


## Split into training and testing

In [4]:
# --- split into training and testing --------------------------------
# the model only ever learns from the training part,
# the test part stays hidden until we score it

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("training rows:", len(X_train))
print("testing rows :", len(X_test))

training rows: 1042
testing rows : 261


## Build the pipeline

Two jobs in one object:
1. turn the text columns into numbers (one-hot encoding)
2. fit the model on those numbers

In [6]:
# this turns the 6 text columns into numbers:
# one extra 0/1 column per category (brand_Apple, brand_Dell, ...)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)

In [7]:
# 2 steps: turn text into numbers -> fit the model
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression()),
])

In [8]:
pipeline.fit(X_train, y_train)
print("model trained")

model trained


## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [9]:
# --- score on the test set -------------------------------------------

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
r2_train = pipeline.score(X_train, y_train)

print("R² on training data :", round(r2_train, 3))
print("R² on test data     :", round(r2, 3))
print("MAE                 :", round(mae), "BDT")
print("RMSE                :", round(rmse), "BDT")

R² on training data : 0.777
R² on test data     : 0.755
MAE                 : 30754 BDT
RMSE                : 44136 BDT


## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [13]:
# --- 5-fold cross-validation on the full data -----------------------
# splits the data 5 ways, trains 5 times, averages the scores

cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print("CV MAE :", round(cv_mae.mean()), "±", round(cv_mae.std()))
print("CV RMSE:", round(cv_rmse.mean()), "±", round(cv_rmse.std()))
print("CV R²  :", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))

CV MAE : 31511 ± 2523
CV RMSE: 43849 ± 4717
CV R²  : 0.741 ± 0.051


### Which features push the price up or down?

In linear regression every feature gets a weight — a big positive weight means *this raises the price*, a big negative one means *this lowers it*.

In [10]:
# which features raise / lower the predicted price the most?

feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()
coefficients = pd.Series(pipeline.named_steps["model"].coef_, index=feature_names)

print("push the price UP:")
print(coefficients.sort_values(ascending=False).head(8).to_string())
print()
print("push the price DOWN:")
print(coefficients.sort_values().head(8).to_string())

push the price UP:
category__brand_Razer         89702.337904
category__brand_LG            74575.440471
category__type_Workstation    68489.879651
category__brand_Google        54963.925467
number__weight_kg             24520.759714
number__cpu_speed_ghz         23450.315721
category__os_Windows          21746.689459
category__brand_Microsoft     20359.289172

push the price DOWN:
category__brand_Chuwi      -59146.636060
category__brand_Fujitsu    -41851.545987
category__brand_Vero       -41508.477669
category__brand_Mediacom   -40443.368225
category__type_Notebook    -34921.727179
category__os_Android       -31168.803806
category__brand_Acer       -26524.882271
category__type_Netbook     -26432.182214


## Try it on 3 real laptops

In [11]:
# --- try it on 3 real laptops from the test set ---------------------

sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    print("real:", real_price, "BDT    predicted:", predicted_price, "BDT")

real: 209000 BDT    predicted: 144396 BDT
real: 143625 BDT    predicted: 139984 BDT
real: 62375 BDT    predicted: 65506 BDT


## Save the scores

Notebook `06_compare_models.ipynb` reads these files.

In [14]:
# --- save the scores (notebook 06 reads these files) ----------------

metrics = {
    "model": "Linear Regression",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_linear_regression.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("saved:", metrics_path)

saved: ..\models\metrics_linear_regression.json


**Next notebook:** `03_decision_tree.ipynb`